In [ ]:
import uuid

uuid_str = "7f288a2e-0ad0-4039-8e59-6c9838d87307"
uuid_obj = uuid.UUID(uuid_str)
print(uuid_obj)


In [ ]:
def __init__(self, dict: Dict) -> None:
    self._dict = dict
    self.source_system = "orders-system-kafka"
    self.order_ns_uuid = uuid.UUID('7f288a2e-0ad0-4039-8e59-6c9838d87307')

def _uuid(self, obj: Any) -> uuid.UUID:
    return uuid.uuid5(namespace=self.order_ns_uuid, name=str(obj))

In [ ]:
from typing import Any, Dict, List
class OrderDdsBuilder:
    def __init__(self, dict: Dict) -> None:
        self._dict = dict
        self.source_system = "orders-system-kafka"
        self.order_ns_uuid = uuid.UUID('7f288a2e-0ad0-4039-8e59-6c9838d87307')

    def _uuid(self, obj: Any) -> uuid.UUID:
        return uuid.uuid5(namespace=self.order_ns_uuid, name=str(obj))

In [ ]:
1+1

In [ ]:
from confluent_kafka import Consumer
import json

def error_callback(err):
    print('Something went wrong: {}'.format(err))

params = {
    'bootstrap.servers': 'rc1a-ujp3lbn9i40iphoa.mdb.yandexcloud.net:9091',
    'security.protocol': 'SASL_SSL',
    'ssl.ca.location': '/Users/Oleg/yandex_pracktikum/CA.pem',
    'sasl.mechanism': 'SCRAM-SHA-512',
    'sasl.username': 'producer_consumer',
    'sasl.password': 'Philips2002',
    'group.id': 'test-consumer1',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': False,
    'error_cb': error_callback,
    'debug': 'all',
}
c = Consumer(params)
c.subscribe(['order-service_orders'])

while True:
    msg = c.poll(timeout=3.0)
    if msg:
        val = msg.value().decode()
        print(val)

%7|1700401287.368|RECV|rdkafka#consumer-2| [thrd:sasl_ssl://rc1a-ujp3lbn9i40iphoa.mdb.yandexcloud.net:9091/boots]: sasl_ssl://rc1a-ujp3lbn9i40iphoa.mdb.yandexcloud.net:9091/1: Received FetchResponse (v11, 82 bytes, CorrId 425, rtt 548.90ms)
%7|1700401287.368|FETCH|rdkafka#consumer-2| [thrd:sasl_ssl://rc1a-ujp3lbn9i40iphoa.mdb.yandexcloud.net:9091/boots]: sasl_ssl://rc1a-ujp3lbn9i40iphoa.mdb.yandexcloud.net:9091/1: Topic order-service_orders [0] MessageSet size 0, error "Success", MaxOffset 6024, LSO 6024, Ver 4/4
%7|1700401287.368|FETCH|rdkafka#consumer-2| [thrd:sasl_ssl://rc1a-ujp3lbn9i40iphoa.mdb.yandexcloud.net:9091/boots]: sasl_ssl://rc1a-ujp3lbn9i40iphoa.mdb.yandexcloud.net:9091/1: Fetch topic order-service_orders [0] at offset 6024 (leader epoch 0, current leader epoch 0, v4)
%7|1700401287.368|FETCH|rdkafka#consumer-2| [thrd:sasl_ssl://rc1a-ujp3lbn9i40iphoa.mdb.yandexcloud.net:9091/boots]: sasl_ssl://rc1a-ujp3lbn9i40iphoa.mdb.yandexcloud.net:9091/1: Fetch 1/1/1 toppar(s)
%7|17004

In [ ]:
!pip3 install confluent_kafka

In [4]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DoubleType
 
# необходимые библиотеки для интеграции Spark с Kafka и PostgreSQL
spark_jars_packages = ",".join(
        [
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
            "org.postgresql:postgresql:42.4.0",
        ]
    )
 
# создаём spark сессию с необходимыми библиотеками в spark_jars_packages для интеграции с Kafka и PostgreSQL
spark = SparkSession.builder \
    .appName("FinalProject") \
    .config("spark.sql.session.timeZone", "UTC") \
    .config("spark.jars.packages", spark_jars_packages) \
    .getOrCreate()
 
# читаем из топика Kafka сообщения с акциями от ресторанов 
transaction_df = (
    spark.read
    .format('kafka')
    .option('kafka.bootstrap.servers', 'rc1a-39in61ae96vcfc9b.mdb.yandexcloud.net:9091')
    .option('kafka.security.protocol', 'SASL_SSL')
    .option('kafka.sasl.jaas.config', 'org.apache.kafka.common.security.scram.ScramLoginModule required username="producer_consumer" password="producer_consumer";') \
    .option('kafka.sasl.mechanism', 'SCRAM-SHA-512')
    .option('kafka.ssl.ca.location', '/Users/Oleg/yandex_pracktikum/CA.pem')
    .option('kafka.sasl.username', 'producer_comsumer')
    .option('kafka.sasl.password', 'producer_consumer')
    #.option('kafka.ssl.truststore.location', '/usr/lib/jvm/java-1.17.0-openjdk-amd64/lib/security/cacerts')
    .option('subscribe', 'order-service_orders')
    .load()
)
transaction_df.printSchema()
transaction_df.show()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



23/11/19 16:47:34 WARN AdminClientConfig: The configuration 'key.deserializer' was supplied but isn't a known config.
23/11/19 16:47:34 WARN AdminClientConfig: The configuration 'value.deserializer' was supplied but isn't a known config.
23/11/19 16:47:34 WARN AdminClientConfig: The configuration 'sasl.username' was supplied but isn't a known config.
23/11/19 16:47:34 WARN AdminClientConfig: The configuration 'max.poll.records' was supplied but isn't a known config.
23/11/19 16:47:34 WARN AdminClientConfig: The configuration 'ssl.ca.location' was supplied but isn't a known config.
23/11/19 16:47:34 WARN AdminClientConfig: The configuration 'enable.auto.commit' was supplied but isn't a known config.
23/11/19 16:47:34 WARN AdminClientConfig: The configuration 'sasl.password' was supplied but isn't a known config.
23/11/19 16:47:34 WARN AdminClientConfig: The configuration 'auto.offset.reset' was supplied but isn't a known config.
23/11/19 16:47:34 ERROR NetworkClient: [AdminClient client

Py4JJavaError: An error occurred while calling o61.showString.
: java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SslAuthenticationException: SSL handshake failed
	at org.apache.kafka.common.internals.KafkaFutureImpl.wrapAndThrow(KafkaFutureImpl.java:45)
	at org.apache.kafka.common.internals.KafkaFutureImpl.access$000(KafkaFutureImpl.java:32)
	at org.apache.kafka.common.internals.KafkaFutureImpl$SingleWaiter.await(KafkaFutureImpl.java:89)
	at org.apache.kafka.common.internals.KafkaFutureImpl.get(KafkaFutureImpl.java:260)
	at org.apache.spark.sql.kafka010.ConsumerStrategy.retrieveAllPartitions(ConsumerStrategy.scala:66)
	at org.apache.spark.sql.kafka010.ConsumerStrategy.retrieveAllPartitions$(ConsumerStrategy.scala:65)
	at org.apache.spark.sql.kafka010.SubscribeStrategy.retrieveAllPartitions(ConsumerStrategy.scala:102)
	at org.apache.spark.sql.kafka010.SubscribeStrategy.assignedTopicPartitions(ConsumerStrategy.scala:113)
	at org.apache.spark.sql.kafka010.KafkaOffs

In [2]:
!pip3 install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 4.9 MB/s eta 0:00:0000:0100:02
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 5.2 MB/s eta 0:00:00a 0:00:01
  Created wheel for pyspark: filename=pyspark-3.5.0-py2.py3-none-any.whl size=317425347 sha256=958f2a8656f8c1eb9abf25f3fac5215bf6cbf32c41e03ab93d401fd983676c69
  Stored in directory: /Users/oleg/Library/Caches/pip/wheels/82/33/7a/7914709b3f4c22fac6942898eaa2103248ac0fde4055377d22
Successfully built pyspark

[notice] A new release of pip available: 22.3.1 -> 23.3.1
[notice] To update, run: pip install --upgrade pip
